# Vortrainiertes YOLO anwenden

Wir verwenden **YOLO26n**, das auf dem COCO-Datensatz mit 80 Klassen vortrainiert ist, und schauen uns an, was in einem Vorhersage-Ergebnis steckt.

## Umgebung

Die Notebooks laufen lokal oder in Google Colab. Für Colab `use_colab = True` setzen und unter *Laufzeit > Laufzeittyp ändern* eine GPU wählen.

Die Version von `ultralytics` ist festgelegt, damit die Notebooks auch in späteren Semestern laufen.

In [ ]:
use_colab = False

if use_colab:
    %pip install -q ultralytics==8.4.215
    from google.colab import drive
    drive.mount("/content/drive")
    base_path = "/content/drive/MyDrive"
else:
    # lokal einmalig installieren: pip install ultralytics==8.4.215
    base_path = ".."

import ultralytics
ultralytics.checks()

In [ ]:
import cv2
import matplotlib.pyplot as plt


def show_image(image_bgr, title="", size=(10, 7)):
    plt.figure(figsize=size)
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

## Modell laden

Beim ersten Aufruf werden die Gewichte (ca. 5 MB) automatisch heruntergeladen.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
print(len(model.names), "Klassen")
print(model.names)

## Vorhersage für ein Bild

`bus.jpg` wird mit `ultralytics` mitgeliefert. Sie können auch ein eigenes Bild angeben.

In [ ]:
from ultralytics.utils import ASSETS

#image = str(ASSETS / "bus.jpg")
image = str("strasse.jpg")
results = model.predict(image, conf=0.25)
r = results[0]                 # ein Results-Objekt pro Bild
show_image(r.plot(), "YOLO26n auf bus.jpg")

## Das Results-Objekt

`r.boxes` enthält alle Detektionen als Tensoren.

In [ ]:
print("Boxen xyxy (Pixel):\n", r.boxes.xyxy)
print("Boxen xywhn (YOLO-Format):\n", r.boxes.xywhn)
print("Confidence:", r.boxes.conf)
print("Klassen-IDs:", r.boxes.cls)
print("Laufzeit in ms:", r.speed)

In [ ]:
import pandas as pd

boxes = r.boxes.cpu().numpy()      # Tensoren ggf. von der GPU holen und in NumPy umwandeln
table = pd.DataFrame({
    "Klasse": [r.names[int(c)] for c in boxes.cls],
    "Confidence": boxes.conf.round(2),
    "x1": boxes.xyxy[:, 0].round(), "y1": boxes.xyxy[:, 1].round(),
    "x2": boxes.xyxy[:, 2].round(), "y2": boxes.xyxy[:, 3].round(),
})
table

## Objekte zählen

In [ ]:
from collections import Counter

Counter(r.names[int(c)] for c in r.boxes.cls)

## Einfluss des Schwellenwerts `conf`

In [ ]:
for conf in [0.1, 0.25, 0.5, 0.75]:
    r = model.predict(image, conf=conf, verbose=False)[0]
    print(f"conf = {conf:.2f}: {len(r.boxes)} Detektionen")

## Mit und ohne NMS

YOLO26 hat zwei Köpfe. Standardmäßig wird der One-to-Many-Head mit NMS verwendet, mit `nms=False` der One-to-One-Head ohne NMS.

In [ ]:
for nms in [True, False]:
    r = model.predict(image, nms=nms, verbose=False)[0]
    print(f"nms = {nms}: {len(r.boxes)} Detektionen, "
          f"Nachverarbeitung {r.speed['postprocess']:.1f} ms")

## Video, Webcam und Tracking

Diese Zellen sind auskommentiert, da sie eine Kamera oder ein Video benötigen.

- `source=0` ist die erste Webcam.
- Mit `stream=True` liefert `predict` einen Generator. So wird nicht jedes Frame im Speicher gehalten.
- `track` vergibt über die Frames hinweg eine **ID je Objekt**.

In [ ]:
# Webcam mit Anzeigefenster, Abbruch mit q
# model.predict(source=0, show=True, conf=0.4)

# Frame für Frame selbst verarbeiten
# for r in model.predict(source=0, stream=True, verbose=False):
#     print(len(r.boxes), "Objekte")

# Tracking in einem Video
# for r in model.track(source="video.mp4", stream=True, persist=True):
#     print(r.boxes.id)